# ESG-Carbon-Telemetry — Colab Validation

**Goal:** Demonstrate async batch ingestion throughput, ARIMA forecasting, and Merkle audit chain integrity.

**Runtime:** CPU only (no GPU needed)

In [1]:
import numpy as np
import pandas as pd
import hashlib
import time
import json
print("Imports loaded.")

Imports loaded.


In [2]:
# Generate synthetic emissions data (monthly, 5 facilities, 3 years)
np.random.seed(42)
facilities = ['Plant_A', 'Plant_B', 'Plant_C', 'Plant_D', 'Plant_E']
months = pd.date_range('2021-01-01', '2023-12-31', freq='MS')

records = []
for fac in facilities:
    base = np.random.uniform(500, 2000)  # tons CO2/month
    seasonal = 200 * np.sin(np.arange(len(months)) * 2 * np.pi / 12)
    trend = np.linspace(0, -100, len(months))  # slight reduction over time
    noise = np.random.normal(0, 50, len(months))
    emissions = base + seasonal + trend + noise
    for i, month in enumerate(months):
        records.append({
            'facility': fac, 'date': month, 'scope': 1,
            'emissions_tons': max(0, emissions[i])
        })

df = pd.DataFrame(records)
print(f"Generated {len(df)} emission records")
print(f"Facilities: {len(facilities)}, Months: {len(months)}")
df.head()

Generated 180 emission records
Facilities: 5, Months: 36


,facility,date,scope,emissions_tons
0,Plant_A,2021-01-01,1,1006.216172
1,Plant_A,2021-02-01,1,1174.898145
2,Plant_A,2021-03-01,1,1243.253038
3,Plant_A,2021-04-01,1,1303.764514
4,Plant_A,2021-05-01,1,1194.542781


In [3]:
# Simulate batch ingestion with flush-on-threshold
BATCH_SIZE = 100
FLUSH_INTERVAL = 0.5  # seconds

class BatchIngester:
    def __init__(self, batch_size=100):
        self.buffer = []
        self.batch_size = batch_size
        self.flush_count = 0
        self.total_ingested = 0
    
    def ingest(self, record):
        self.buffer.append(record)
        if len(self.buffer) >= self.batch_size:
            self.flush()
    
    def flush(self):
        # Simulate COPY-based bulk write (just measure throughput)
        self.total_ingested += len(self.buffer)
        self.flush_count += 1
        self.buffer = []

# Benchmark: row-by-row vs batch
ingester = BatchIngester(batch_size=BATCH_SIZE)

t0 = time.time()
for _, row in df.iterrows():
    ingester.ingest(row.to_dict())
ingester.flush()  # final partial batch
batch_time = time.time() - t0

# Simulate row-by-row (just timing the iteration overhead)
t0 = time.time()
for _, row in df.iterrows():
    _ = row.to_dict()  # simulate per-row processing
row_time = time.time() - t0

print(f"=== Ingestion Benchmark ===")
print(f"Records: {ingester.total_ingested}")
print(f"Batch flushes: {ingester.flush_count}")
print(f"Batch time: {batch_time:.3f}s ({ingester.total_ingested/batch_time:.0f} records/s)")
print(f"Row-by-row time: {row_time:.3f}s ({len(df)/row_time:.0f} records/s)")
print(f"Speedup: {row_time/batch_time:.1f}x")

=== Ingestion Benchmark ===
Records: 180
Batch flushes: 2
Batch time: 0.005s (35585 records/s)
Row-by-row time: 0.005s (36441 records/s)
Speedup: 1.0x


In [4]:
# ARIMA Forecasting
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error

# Forecast for Plant_A
plant_a = df[df['facility'] == 'Plant_A'].set_index('date')['emissions_tons']

# Train on first 30 months, test on last 6
train_ts = plant_a.iloc[:30]
test_ts = plant_a.iloc[30:]

t0 = time.time()
model = ARIMA(train_ts, order=(2, 1, 1))
fitted = model.fit()
forecast = fitted.forecast(steps=len(test_ts))
arima_time = time.time() - t0

mae = mean_absolute_error(test_ts.values, forecast.values)
mape = np.mean(np.abs((test_ts.values - forecast.values) / test_ts.values)) * 100

print(f"\n=== ARIMA(2,1,1) Forecast ===")
print(f"Train: {len(train_ts)} months | Test: {len(test_ts)} months")
print(f"MAE: {mae:.1f} tons CO2")
print(f"MAPE: {mape:.1f}%")
print(f"Fit time: {arima_time:.3f}s")
print(f"\nActual vs Forecast (last 6 months):")
for actual, pred in zip(test_ts.values[:6], forecast.values[:6]):
    print(f"  Actual: {actual:.0f} | Forecast: {pred:.0f} | Error: {abs(actual-pred):.0f}")

C:\Users\pooja\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\Users\pooja\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\Users\pooja\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
C:\Users\pooja\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
C:\Users\po


=== ARIMA(2,1,1) Forecast ===
Train: 30 months | Test: 6 months
MAE: 167.8 tons CO2
MAPE: 20.1%
Fit time: 0.299s

Actual vs Forecast (last 6 months):
  Actual: 965 | Forecast: 1056 | Error: 91
  Actual: 893 | Forecast: 1048 | Error: 155
  Actual: 734 | Forecast: 1043 | Error: 309
  Actual: 822 | Forecast: 1042 | Error: 220
  Actual: 930 | Forecast: 1041 | Error: 111
  Actual: 921 | Forecast: 1041 | Error: 120


In [5]:
# Merkle Hash-Chain Audit Trail
class MerkleAuditChain:
    def __init__(self):
        self.chain = []
        self.root = hashlib.sha256(b'genesis').hexdigest()
    
    def append(self, record: dict):
        payload = json.dumps(record, sort_keys=True, default=str)
        prev_hash = self.chain[-1]['hash'] if self.chain else self.root
        block_data = f"{prev_hash}{payload}"
        block_hash = hashlib.sha256(block_data.encode()).hexdigest()
        self.chain.append({'record': record, 'prev': prev_hash, 'hash': block_hash})
        return block_hash
    
    def verify(self) -> bool:
        for i, block in enumerate(self.chain):
            prev_hash = self.chain[i-1]['hash'] if i > 0 else self.root
            payload = json.dumps(block['record'], sort_keys=True, default=str)
            expected = hashlib.sha256(f"{prev_hash}{payload}".encode()).hexdigest()
            if expected != block['hash']:
                return False
        return True

# Build chain from all records
chain = MerkleAuditChain()
t0 = time.time()
for _, row in df.iterrows():
    chain.append(row.to_dict())
chain_time = time.time() - t0

# Verify integrity
t0 = time.time()
is_valid = chain.verify()
verify_time = time.time() - t0

print(f"\n=== Merkle Audit Chain ===")
print(f"Chain length: {len(chain.chain)} blocks")
print(f"Build time: {chain_time:.3f}s")
print(f"Verify time: {verify_time:.3f}s")
print(f"Chain valid: {is_valid}")
print(f"Root hash: {chain.chain[-1]['hash'][:16]}...")

# Tamper test
chain.chain[50]['record']['emissions_tons'] = 9999
is_valid_after_tamper = chain.verify()
print(f"After tampering record #50: valid={is_valid_after_tamper} (should be False)")


=== Merkle Audit Chain ===
Chain length: 180 blocks
Build time: 0.009s
Verify time: 0.001s
Chain valid: True
Root hash: a03991db1ea15f72...
After tampering record #50: valid=False (should be False)


In [6]:
# Summary
import os
results = {
    "ingestion": {
        "records": int(ingester.total_ingested),
        "batch_throughput_rps": round(ingester.total_ingested / batch_time),
        "row_throughput_rps": round(len(df) / row_time),
        "speedup": round(row_time / batch_time, 1)
    },
    "forecast": {
        "model": "ARIMA(2,1,1)",
        "mae_tons": round(mae, 1),
        "mape_pct": round(mape, 1),
        "fit_time_s": round(arima_time, 3)
    },
    "audit_chain": {
        "blocks": len(chain.chain),
        "build_time_s": round(chain_time, 3),
        "verify_time_s": round(verify_time, 3),
        "tamper_detected": not is_valid_after_tamper
    }
}
print(json.dumps(results, indent=2))
os.makedirs('results', exist_ok=True)
with open('results/colab_validation.json', 'w') as f:
    json.dump(results, f, indent=2)

{
  "ingestion": {
    "records": 180,
    "batch_throughput_rps": 35585,
    "row_throughput_rps": 36441,
    "speedup": 1.0
  },
  "forecast": {
    "model": "ARIMA(2,1,1)",
    "mae_tons": 167.8,
    "mape_pct": 20.1,
    "fit_time_s": 0.299
  },
  "audit_chain": {
    "blocks": 180,
    "build_time_s": 0.009,
    "verify_time_s": 0.001,
    "tamper_detected": true
  }
}
